# NB01: Data Collection

This notebook collects finished match results for the 2025-26 season of Europe's "big five" leagues from the [football-data.org](https://www.football-data.org/) free API, to investigate whether home advantage is equally strong across leagues.

In [1]:
import json
import time
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env into the environment, so the key below never has to be typed into the notebook

API_KEY = os.getenv("FOOTBALL_DATA_API_KEY")
BASE_URL = "https://api.football-data.org/v4"
HEADERS = {"X-Auth-Token": API_KEY}  # football-data.org authenticates via this header, not a query parameter

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)  # parents=True creates data/ too if it doesn't exist yet; exist_ok=True means re-running this cell won't error

## Choosing my leagues and season

I'm comparing the Premier League, La Liga, Bundesliga, Serie A and Ligue 1 - the five leagues most commonly grouped as Europe's strongest domestic competitions. Restricting to one completed season (2025-26) means every league has a full, fixed set of matchdays and no in-progress fixtures that would bias the home/away split.

football-data.org's free tier gives access to all five competitions and labels a season by its starting year, so `season=2025` here means the 2025-26 season, which finished in May 2026.

In [2]:
# The "big five" European leagues, identified by their football-data.org competition codes
COMPETITIONS = {
    "PL": "Premier League (England)",
    "PD": "La Liga (Spain)",
    "BL1": "Bundesliga (Germany)",
    "SA": "Serie A (Italy)",
    "FL1": "Ligue 1 (France)",
}

SEASON = 2025

## The shape of the API response

Before writing the loop that collects all five leagues, I make one request for a single competition and look at what actually comes back. This is faster than guessing field names from the documentation alone, and it tells me exactly which fields NB02 will need to pull out later.

In [3]:
# One request, nothing saved yet - just to see the response shape before writing the full loop below.
sample_response = requests.get(
    f"{BASE_URL}/competitions/PL/matches",
    headers=HEADERS,
    params={"season": SEASON, "status": "FINISHED"},
)
sample_response.raise_for_status()
sample_payload = sample_response.json()

print("Top-level keys:", list(sample_payload.keys()))
print(f"Number of matches on this page: {len(sample_payload['matches'])}")
print("\nOne match:")
print(json.dumps(sample_payload["matches"][0], indent=2))

Top-level keys: ['filters', 'resultSet', 'competition', 'matches']
Number of matches on this page: 380

One match:
{
  "area": {
    "id": 2072,
    "name": "England",
    "code": "ENG",
    "flag": "https://crests.football-data.org/770.svg"
  },
  "competition": {
    "id": 2021,
    "name": "Premier League",
    "code": "PL",
    "type": "LEAGUE",
    "emblem": "https://crests.football-data.org/PL.png"
  },
  "season": {
    "id": 2403,
    "startDate": "2025-08-15",
    "endDate": "2026-05-24",
    "currentMatchday": 38,
    "winner": null
  },
  "id": 537785,
  "utcDate": "2025-08-15T19:00:00Z",
  "status": "FINISHED",
  "matchday": 1,
  "stage": "REGULAR_SEASON",
  "group": null,
  "lastUpdated": "2026-06-07T20:20:25Z",
  "homeTeam": {
    "id": 64,
    "name": "Liverpool FC",
    "shortName": "Liverpool",
    "tla": "LIV",
    "crest": "https://crests.football-data.org/64.png"
  },
  "awayTeam": {
    "id": 1044,
    "name": "AFC Bournemouth",
    "shortName": "Bournemouth",
    

The response is a dictionary with four top-level keys - `filters`, `resultSet`, `competition` and `matches` - of which `matches` is the one that matters: a list with one entry per fixture, and (unlike the endpoint I'll paginate against in other projects) the whole season comes back in a single request, no paging needed.

Each match carries far more than I need: full team crests, referee names, an `odds` field that's just a placeholder on the free tier, competition metadata repeated on every single match. The fields NB02 will actually use are `homeTeam.name`, `awayTeam.name`, `score.fullTime.home`, `score.fullTime.away`, `utcDate`, `matchday` and `status` - everything else gets ignored when I flatten this into a table.

## Fetching finished matches

Now that I know the shape of a single response, I fetch every league. For each competition I pull only `status=FINISHED` matches for the 2025-26 season and save the raw API response as-is under `data/raw/`, one JSON file per league. This keeps the original source data untouched, before any cleaning happens in NB02.

The free plan allows 10 requests per minute; five sequential calls comfortably fit, but I still add a short pause between requests to stay well under the limit.

In [4]:
for code, name in COMPETITIONS.items():
    url = f"{BASE_URL}/competitions/{code}/matches"
    params = {"season": SEASON, "status": "FINISHED"}

    response = requests.get(url, headers=HEADERS, params=params)
    response.raise_for_status()  # stop immediately on a bad response rather than saving a partial/error payload
    payload = response.json()

    out_path = RAW_DIR / f"{code}_matches_{SEASON}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)  # ensure_ascii=False keeps accented team names (e.g. "Atlético") readable in the file rather than escaped

    print(f"{name} ({code}): {len(payload['matches'])} finished matches saved to {out_path}")
    time.sleep(6)  # free tier allows 10 requests/minute; 6s between calls keeps me well under that

Premier League (England) (PL): 380 finished matches saved to data\raw\PL_matches_2025.json


La Liga (Spain) (PD): 380 finished matches saved to data\raw\PD_matches_2025.json


Bundesliga (Germany) (BL1): 306 finished matches saved to data\raw\BL1_matches_2025.json


Serie A (Italy) (SA): 380 finished matches saved to data\raw\SA_matches_2025.json


Ligue 1 (France) (FL1): 305 finished matches saved to data\raw\FL1_matches_2025.json


## Checking what we collected

Quick sanity check: reload each saved file and confirm the match count and date range look right (a full season, not a partial one).

In [5]:
for code, name in COMPETITIONS.items():
    path = RAW_DIR / f"{code}_matches_{SEASON}.json"
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)

    matches = payload["matches"]
    dates = sorted(m["utcDate"] for m in matches)  # sorting lets me read off the first and last match date without scanning the whole list by eye
    print(f"{name}: {len(matches)} matches, {dates[0][:10]} to {dates[-1][:10]}")

Premier League (England): 380 matches, 2025-08-15 to 2026-05-24
La Liga (Spain): 380 matches, 2025-08-15 to 2026-05-24
Bundesliga (Germany): 306 matches, 2025-08-22 to 2026-05-16
Serie A (Italy): 380 matches, 2025-08-23 to 2026-05-24
Ligue 1 (France): 305 matches, 2025-08-15 to 2026-05-17


| League | Matches | Date range |
|---|---|---|
| Premier League | 380 | 2025-08-15 to 2026-05-24 |
| La Liga | 380 | 2025-08-15 to 2026-05-24 |
| Bundesliga | 306 | 2025-08-22 to 2026-05-16 |
| Serie A | 380 | 2025-08-23 to 2026-05-24 |
| Ligue 1 | 305 | 2025-08-15 to 2026-05-17 |
| **Total** | **1,751** | |

**These are complete, full-season counts, not partial pulls.** 380 matches is exactly what a 20-team, double round-robin league produces (20 x 19 = 380); 306 and 305 are the equivalent for the 18-team Bundesliga and Ligue 1. Every league also spans mid-August to mid/late-May, a full season rather than a truncated one - so the home/away comparison in NB03 rests on complete seasons for all five leagues, not a mix of full and partial ones.

## Sources and tools

**Data source**
- [football-data.org](https://www.football-data.org/), free tier. Match results for Europe's major football competitions, no cost or card required for the tier used here.

**API documentation**
- football-data.org API documentation: https://www.football-data.org/documentation/quickstart - used to find the free-tier competition codes, the `season` and `status` query parameters, and the `X-Auth-Token` authentication header.

**Python libraries**
- `requests`, for the HTTP calls: https://requests.readthedocs.io/
- `python-dotenv`, for loading the API key from `.env` without hardcoding it in the notebook: https://pypi.org/project/python-dotenv/

**AI tools**
- I used Claude to write the collection code in this notebook: the peek request that shows the response shape, the loop over the five competitions (including handling football-data.org's season-labelled-by-start-year convention and the pause between requests to respect the free-tier rate limit), and the sanity-check cell. I chose which leagues and season to use, ran the API key against the live endpoint to confirm all five competitions were reachable on the free tier, and reviewed the printed match counts and date ranges myself before treating the data as ready for NB02.